# 评估集工程

前置知识：C5 系统评估与优化、本章 Notebook 2-3

本节目标：掌握评估集的分桶构建方法，学会用 LLM 自动生成评估问题，构建可维护的评估数据集。

## 一、评估集的三种类型

在生产环境中，评估集不是一次性的，而是需要持续维护和更新。按用途分为三类：

### 1. Golden Set（核心回归集）
- **定位**：每次改动必跑的核心集合
- **规模**：50-200 条
- **特点**：覆盖所有关键场景，经过人工审核确认，质量最高
- **维护**：只增不删，每条都有明确的预期答案

### 2. Regression Set（回归测试集）
- **定位**：历史 Bad Case 的累积
- **规模**：持续增长
- **特点**：每次发现新的 Bad Case 就加入，防止同样的问题反复出现
- **维护**：修复后保留作为回归用例

### 3. Hard Set（挑战测试集）
- **定位**：边界场景和困难用例
- **规模**：20-50 条
- **特点**：专门测试系统的弱点，如多跳推理、模糊问题、拒答场景
- **维护**：定期更新，反映当前系统的瓶颈

## 二、分桶策略

把评估问题按类型分桶，每个桶测试系统的不同能力：

| 桶类型 | 含义 | 示例 | 期望行为 |
|--------|------|------|----------|
| **factual** | 事实问答 | "什么是信息增益？" | 准确引用文档内容 |
| **reasoning** | 推理题 | "为什么 SVM 要引入核函数？" | 基于文档做推理 |
| **ambiguous** | 模糊问题 | "哪个算法最好？" | 给出有条件的回答 |
| **unanswerable** | 拒答场景 | "GPT-5 的参数量？" | 明确告知无法回答 |

分桶的目的是**找出系统的弱点桶**——如果某个桶的得分明显低于其他桶，说明系统在该能力上存在短板。

In [ ]:
import os
import json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from modelscope import snapshot_download
model_dir = snapshot_download('BAAI/bge-small-zh-v1.5', cache_dir='./models')
print(f"Embedding 模型路径: {model_dir}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_models import ChatZhipuAI

embedding = HuggingFaceEmbeddings(model_name=model_dir)
api_key = os.environ.get("ZHIPUAI_API_KEY")
llm = ChatZhipuAI(model="glm-4-flash", temperature=0.0, api_key=api_key)
print("Embedding 和 LLM 初始化完成")

In [ ]:
import re
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

pdf_path = "../3. 索引阶段/data/pumpkin_book.pdf"
persist_dir = "./chroma_db"

def clean_text(text: str) -> str:
    text = re.sub(r'→_→\n.*?←_←', '', text, flags=re.DOTALL)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def build_vectorstore(pdf_path, embedding, persist_directory="./chroma_db"):
    """构建或加载向量库，已有则复用"""
    if os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"发现已存在的向量库: {persist_directory}，正在加载...")
        try:
            vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding)
            count = vectorstore._collection.count()
            print(f"✅ 加载成功！共 {count} 个文档块")
            return vectorstore
        except Exception as e:
            print(f"⚠️ 加载失败 ({e})，将重新构建...")
    print("开始构建向量库...")
    loader = PyMuPDFLoader(pdf_path)
    pdf_pages = loader.load()
    data_pages = pdf_pages[13:-13]
    for page in data_pages:
        page.page_content = clean_text(page.page_content)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    splits = text_splitter.split_documents(data_pages)
    vectorstore = Chroma.from_documents(documents=splits, embedding=embedding, persist_directory=persist_directory)
    print(f"✅ 向量库构建完成并保存至 {persist_directory}，共 {len(splits)} 个文档块")
    return vectorstore

vectorstore = build_vectorstore(pdf_path, embedding, persist_dir)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_template(
    "根据以下上下文回答问题。如果上下文中没有相关信息，请说'根据已有资料无法回答'。\n\n"
    "上下文：\n{context}\n\n问题：{question}\n\n回答："
)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)
print("RAG Pipeline 构建完成")

## 三、用 LLM 自动生成评估问题

手动编写评估集耗时耗力，我们可以利用 LLM 基于文档内容自动生成不同类型的评估问题。

**策略**：给定一段文档文本和目标题目类型，让 LLM 生成问题 + 标准答案。

In [ ]:
def generate_eval_questions(text: str, bucket: str, num_questions: int = 2) -> list:
    """基于文档文本自动生成指定类型的评估问题"""
    bucket_instructions = {
        "factual": "生成事实性问题，答案可以直接从文本中找到。",
        "reasoning": "生成需要推理的问题，需要理解文本内容并进行逻辑推导才能回答。",
        "ambiguous": "生成有一定模糊性的问题，文本中没有唯一明确的答案。",
    }

    instruction = bucket_instructions.get(bucket, bucket_instructions["factual"])

    gen_prompt = ChatPromptTemplate.from_template(
        "你是一个评估集构建专家。请根据以下文本内容，生成 {num} 个评估问题。\n\n"
        "要求：{instruction}\n\n"
        "文本内容：\n{text}\n\n"
        "请严格按以下 JSON 格式返回（只返回 JSON 数组，不要返回其他内容）：\n"
        '[{{"question": "问题内容", "ground_truth": "标准答案"}}]'
    )

    chain = gen_prompt | llm | StrOutputParser()
    response = chain.invoke({
        "num": num_questions,
        "instruction": instruction,
        "text": text[:1500]
    })

    try:
        response = response.strip()
        if response.startswith("```"):
            response = response.split("\n", 1)[1].rsplit("```", 1)[0]
        questions = json.loads(response)
        for q in questions:
            q["bucket"] = bucket
            q["difficulty"] = "auto"
        return questions
    except json.JSONDecodeError:
        print(f"JSON 解析失败，原始响应：{response[:200]}")
        return []

sample_text = splits[10].page_content
generated = generate_eval_questions(sample_text, "factual", 2)
for q in generated:
    print(f"[{q['bucket']}] {q['question']}")
    print(f"  答案：{q['ground_truth'][:80]}...")
    print()

In [ ]:
with open("./data/eval_questions.json", "r", encoding="utf-8") as f:
    eval_questions = json.load(f)

print(f"加载了 {len(eval_questions)} 条评估问题\n")

from collections import Counter
bucket_counts = Counter(q["bucket"] for q in eval_questions)
for bucket, count in sorted(bucket_counts.items()):
    print(f"  {bucket}: {count} 条")

In [ ]:
from tqdm import tqdm
import time

eval_results = []
for item in tqdm(eval_questions, desc="评估中"):
    docs = retriever.invoke(item["question"])
    answer = rag_chain.invoke(item["question"])
    eval_results.append({
        **item,
        "answer": answer,
        "contexts": [doc.page_content for doc in docs],
    })
    time.sleep(0.5)

print(f"已完成 {len(eval_results)} 条评估")

In [ ]:
def simple_judge(question: str, answer: str, ground_truth: str, context: str) -> dict:
    """简易 Judge：综合评分"""
    judge_prompt = ChatPromptTemplate.from_template(
        "你是一个 RAG 系统评估员。请评估系统的回答质量。\n\n"
        "用户问题：{question}\n"
        "标准答案：{ground_truth}\n"
        "检索到的上下文：{context}\n"
        "系统回答：{answer}\n\n"
        "请从以下三个维度分别打分（0-10 分），按 JSON 格式返回：\n"
        '{{"relevance": <答案相关性分数>, "faithfulness": <忠实度分数>, "correctness": <正确性分数>}}'
    )
    chain = judge_prompt | llm | StrOutputParser()
    response = chain.invoke({
        "question": question, "answer": answer,
        "ground_truth": ground_truth, "context": context[:2000]
    })
    try:
        response = response.strip()
        if response.startswith("```"):
            response = response.split("\n", 1)[1].rsplit("```", 1)[0]
        scores = json.loads(response)
        return {k: max(0, min(10, int(v))) for k, v in scores.items()}
    except (json.JSONDecodeError, ValueError):
        return {"relevance": 5, "faithfulness": 5, "correctness": 5}

scored_results = []
for i, r in enumerate(tqdm(eval_results, desc="Judge 评分")):
    ctx = "\n\n".join(r["contexts"][:3])
    scores = simple_judge(r["question"], r["answer"], r["ground_truth"], ctx)
    scored_results.append({**r, "scores": scores})
    time.sleep(1)

print(f"评分完成，共 {len(scored_results)} 条")

In [ ]:
import numpy as np

bucket_scores = {}
for r in scored_results:
    bucket = r["bucket"]
    if bucket not in bucket_scores:
        bucket_scores[bucket] = {"relevance": [], "faithfulness": [], "correctness": []}
    for key in ["relevance", "faithfulness", "correctness"]:
        bucket_scores[bucket][key].append(r["scores"][key])

print("=" * 70)
print("分桶评估报告")
print("=" * 70)
print(f"{'桶类型':<15} {'数量':>4} {'相关性':>8} {'忠实度':>8} {'正确性':>8} {'均分':>8}")
print("-" * 70)

for bucket in ["factual", "reasoning", "ambiguous", "unanswerable"]:
    if bucket in bucket_scores:
        s = bucket_scores[bucket]
        n = len(s["relevance"])
        rel = np.mean(s["relevance"])
        faith = np.mean(s["faithfulness"])
        corr = np.mean(s["correctness"])
        avg = (rel + faith + corr) / 3
        print(f"{bucket:<15} {n:>4} {rel:>8.1f} {faith:>8.1f} {corr:>8.1f} {avg:>8.1f}")

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

buckets = [b for b in ["factual", "reasoning", "ambiguous", "unanswerable"] if b in bucket_scores]
metrics = ["relevance", "faithfulness", "correctness"]
x = np.arange(len(buckets))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
for i, metric in enumerate(metrics):
    values = [np.mean(bucket_scores[b][metric]) for b in buckets]
    ax.bar(x + i * width, values, width, label=metric)

ax.set_ylabel('分数 (0-10)')
ax.set_title('分桶评估得分对比')
ax.set_xticks(x + width)
ax.set_xticklabels(buckets)
ax.legend()
ax.set_ylim(0, 11)

plt.tight_layout()
plt.savefig("./figures/bucket_scores.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("弱点分析")
print("=" * 50)

bucket_avg = {}
for bucket in buckets:
    s = bucket_scores[bucket]
    avg = np.mean([np.mean(s[m]) for m in metrics])
    bucket_avg[bucket] = avg

weakest = min(bucket_avg, key=bucket_avg.get)
print(f"\n最弱桶：{weakest}（均分 {bucket_avg[weakest]:.1f}）")

advice = {
    "factual": "事实问答得分低 → 检查检索召回率，可能需要优化 embedding 或增加文档覆盖",
    "reasoning": "推理题得分低 → 考虑使用 Chain-of-Thought prompt 或增大上下文窗口",
    "ambiguous": "模糊问题得分低 → 优化 prompt，增加对不确定性的处理指导",
    "unanswerable": "拒答场景得分低 → 在 prompt 中明确拒答策略，增加'无法回答'的示例",
}
print(f"优化建议：{advice.get(weakest, '请针对具体问题分析')}")

## 四、评估集的持续更新

评估集不是一次性产物，需要持续维护：

### 更新触发时机
1. **新增功能**：每新增一个功能，至少添加 5 条对应的评估问题
2. **发现 Bad Case**：每个 Bad Case 都应该加入 Regression Set
3. **定期扩充**：每月用 LLM 基于新数据自动生成候选问题，人工审核后入库

### 质量控制
1. 每条新问题都需要人工审核标准答案
2. 定期清理过时的评估问题（如数据源已更新）
3. 保持各桶的比例均衡，避免某个桶过多或过少

## 五、小结

本节介绍了评估集工程的核心方法：

### 关键要点
1. **三种评估集各有用途**：Golden Set 保质量、Regression Set 防回归、Hard Set 测边界
2. **分桶策略找弱点**：按问题类型分桶，找出系统的能力短板
3. **LLM 辅助生成**：用 LLM 生成候选问题，人工审核确保质量
4. **持续更新**：评估集是活的，需要跟随系统演进

### 实践建议
1. **从 50 条开始**：不要追求一步到位，先建一个小而精的 Golden Set
2. **每个桶至少 5 条**：确保各桶都有足够样本
3. **保存元数据**：记录每条问题的来源页码、创建时间、创建方式（手动/自动）
4. **版本管理**：评估集放入 Git 版本控制，每次修改都可追溯

### 参考文献
- [LLM-based Evaluation Set Generation](https://docs.ragas.io/en/latest/concepts/testset_generation.html)